# Translation weight effect — liquid water

Compare cross sections for the reference and samples at 295 K. Translation sampling reuses the normalized vibrational spectrum.

## 0. Choose the engine

**Required setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ENGINE` | `"ncrystal"` | Select `"njoy"` or `"ncrystal"` for this run. |

After changing the engine, restart the kernel and run all cells.

In [ ]:
AVAILABLE_ENGINE = ("njoy", "ncrystal")
AVAILABLE_ENGINE_LABEL = {
    "njoy": "NJOY",
    "ncrystal": "NCrystal"
}

In [ ]:
ENGINE = "ncrystal"
if ENGINE not in AVAILABLE_ENGINE:
    raise ValueError('Choose ENGINE = "njoy" or "ncrystal".')
ENGINE_LABEL = AVAILABLE_ENGINE_LABEL[ENGINE]

## 1. Locate TARA

The next cell finds TARA and imports the APIs for the chosen engine.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "core" / "config.py").is_file()
     and (path / "processing" / "pipelines" / "ace_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the TARA repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import parallel_execution
from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler

if ENGINE == "njoy":
    from input_output.leapr.core import LeaprFile
    from input_output.leapr.material_adapter import material_model_from_leapr
elif ENGINE == "ncrystal":
    from input_output.ncrystal.core import NCrystalFile
    from input_output.ncrystal.liquids import load_liquid_ncmat, parse_custom_liquids

## 2. Select the required inputs

| Setting | Example value | What it controls |
| --- | --- | --- |
| `BASE_LEAPR_FILE` | `tests/fixtures/tsl-HinH2O.leapr` | NJOY material template. |
| `BASE_NCMAT_FILE` | `tests/fixtures/liquid_water_295K_illustrative.ncmat` | NCrystal material. |
| `BASE_ENDF_FILE` | `tests/fixtures/n_1-H-001g.jeff` | Neutron evaluation for the NJOY path. Override with `TARA_BASE_ENDF` if needed. |
| `NJOY_EXECUTABLE` | `NJOY_EXEC` or `njoy` on `PATH` | Executable for the NJOY path. |

Edit paths only if your checkout or NJOY installation uses different locations.

NCrystal liquid calculations require the Liquids plugin.

In [ ]:
fixtures = PROJECT_ROOT / "tests" / "fixtures"
BASE_ENDF_FILE = Path(os.environ.get("TARA_BASE_ENDF", str(fixtures / "n_1-H-001g.jeff")))
njoy_path = shutil.which(os.environ.get("NJOY_EXEC") or "njoy")
NJOY_EXECUTABLE = Path(njoy_path) if njoy_path else None

if ENGINE == "njoy":
    if not BASE_ENDF_FILE.is_file() or BASE_ENDF_FILE.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty neutron ENDF: {BASE_ENDF_FILE}")
    if NJOY_EXECUTABLE is None:
        raise FileNotFoundError("Set NJOY_EXEC or add njoy to PATH.")

    print("Neutron ENDF:", BASE_ENDF_FILE)
    print("NJOY:", NJOY_EXECUTABLE)

In [ ]:
if ENGINE == "njoy":
    BASE_LEAPR_FILE = fixtures / "tsl-HinH2O.leapr"
    if not BASE_LEAPR_FILE.is_file():
        raise FileNotFoundError(f"LEAPR template missing: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    BASE_NCMAT_FILE = fixtures / "liquid_water_295K_illustrative.ncmat"
    if not BASE_NCMAT_FILE.is_file():
        raise FileNotFoundError(f"NCMAT template missing: {BASE_NCMAT_FILE}")

## 3. Load the liquid water reference

**Fixed settings for this example**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `TEMPERATURE_K` | `295` | Evaluation temperature in kelvin. |
| `ATOM_LABEL` | `"H"` | Species supplying the reference spectrum. |

In [ ]:
TEMPERATURE_K = 295.0
ATOM_LABEL = "H"
print(f"Engine: {ENGINE_LABEL}")
print(f"temperature: {TEMPERATURE_K:g} K")

if ENGINE == "njoy":
    template = LeaprFile.from_file(str(BASE_LEAPR_FILE))
    leapr_template = template

    material = material_model_from_leapr(
        template, state="liquid", temperature=TEMPERATURE_K,
        principal_label=ATOM_LABEL, secondary_label="O",
    )
    if (material.principal.multiplicity != 2 or len(material.secondary) != 1
            or material.secondary[0].multiplicity != 1):
        raise ValueError("Expected the CAB H-in-water LEAPR material.")
    dynamics = material.dynamics
    reference_spectrum = dynamics.continuous
    raw_pdos = reference_spectrum.pdos
    vibrational_weight = dynamics.continuous_weight + sum(
        oscillator.weight for oscillator in dynamics.oscillators
    )
    translation_weight = dynamics.translation_weight
    print(f"LEAPR file: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    source_text = load_liquid_ncmat(BASE_NCMAT_FILE)
    liquid_entries = parse_custom_liquids(source_text)
    liquid_settings = next((entry for entry in liquid_entries if entry.atom == ATOM_LABEL), None)
    if liquid_settings is None:
        raise ValueError(f"No custom liquid settings for {ATOM_LABEL!r}.")
    vibrational_weight = liquid_settings.ws
    translation_weight = liquid_settings.wt

    material = NCrystalFile.from_file(str(BASE_NCMAT_FILE))
    raw_pdos = material.pdos_for_atom(ATOM_LABEL)
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"NCMAT file: {BASE_NCMAT_FILE}")

print(f"Vibrational weight: {vibrational_weight:g}")
print(f"Translation weight: {translation_weight:g}")

## 4. Set the translation sampling controls

These bounds define a sensitivity study. They are relative to the reference values.

**Required settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_WEIGHT_SHIFT_PCT` | `10.0` | Maximum relative change in translation weight, as a percentage. |
| `N_SAMPLES` | `3` | Stochastic realizations in addition to the reference. |

**Optional settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RANDOM_SEED` | `42` | Reproduces the Latin hypercube draws. |

In [ ]:
MAX_WEIGHT_SHIFT_PCT = 10.0
N_SAMPLES = 3
RANDOM_SEED = 42

In [ ]:
from core.config import DiffusionSamplingConfig

diffusion_config = DiffusionSamplingConfig(
    enabled=True,
    mode="weight_fixed_d",
    max_weight_shift_pct=MAX_WEIGHT_SHIFT_PCT,
    weight_compensation="continuous_only",
)

## 5. Generate the samples

Each realization reuses the reference vibrational shape. The next cell
prints the sampled parameters and their factors relative to the reference.

In [ ]:
from IPython.display import display
from core.sampler.diffusion_sampler import DiffusionSampler

if ENGINE == "ncrystal":
    import NCrystal as NC
    from scipy.constants import physical_constants

    from core.data.material_scattering import (
        MaterialScatteringModel,
        ScatteringSpecies,
        SpeciesDynamics,
    )

    hydrogen = liquid_settings
    neutron_mass = physical_constants["neutron mass in u"][0]
    material = MaterialScatteringModel(
        identifier=BASE_NCMAT_FILE.stem,
        state="liquid",
        temperature_k=TEMPERATURE_K,
        principal=ScatteringSpecies(ATOM_LABEL, 2, NC.atomDB(ATOM_LABEL).averageMassAMU() / neutron_mass),
        dynamics=SpeciesDynamics(
            continuous=reference_spectrum,
            continuous_weight=hydrogen.ws,
            translation_weight=hydrogen.wt,
            translation_model="egelstaff_schofield",
            translation_parameter=hydrogen.c,
        ),
        secondary=(ScatteringSpecies("O", 1, NC.atomDB("O").averageMassAMU() / neutron_mass),),
    )

reference = material.dynamics
if reference.translation_model != "egelstaff_schofield":
    raise ValueError("Select a reference with Egelstaff–Schofield translation.")
print(f"Reference diffusion parameter (c): {reference.translation_parameter:.7g}")
print(f"Reference scaled diffusivity: {reference.translation_weight * reference.translation_parameter:.7g}")

In [ ]:
ensemble = DiffusionSampler(
    material, diffusion_config, seed=RANDOM_SEED
).generate_ensemble(N_SAMPLES)
realizations = {"sample_000": ensemble.reference}
realizations.update(
    {f"sample_{index + 1:03d}": ensemble[index] for index in range(len(ensemble))}
)
info = ensemble.sample_info.copy()
info.index = list(realizations)[1:]
info.index.name = "Sample"
print("sample_000 is the reference. Factors below are relative to that material.")


if ENGINE == "njoy":
    translation_label = "twt"
elif ENGINE == "ncrystal":
    translation_label = "wt"

scaled_label = f"{translation_label} × c"
column_labels = {
    "translation_weight": translation_label,
    "diffusion_parameter_c": "c",
    "scaled_diffusion_d": scaled_label,
    "diffusivity_factor": "Diffusivity factor",
    "weight_factor": "Weight factor",
}
display(info[list(column_labels)].rename(columns=column_labels))

## 6. Calculate the thermal scattering cross sections

**NJOY**

Each calculation serializes the complete material realization, including
its `twt` and `c` on LEAPR Card 13. NJOY runs LEAPR and THERMR using the
native oscillator representation. For this CAB fixture there is no
separate thermal elastic contribution. MF3/MT221 therefore gives the
thermal scattering cross section for this comparison.

**NCrystal**

Each sampled `@CUSTOM_LIQUIDS` block is written to a temporary material
file. NCrystal evaluates the native scattering cross section per atom
for the full illustrative water material, including the oxygen free-gas
contribution. The sample-to-reference differences isolate changes in
the hydrogen settings. The NJOY notebooks report the hydrogen thermal
scattering reaction.

**Optional settings**

**NJOY**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `THERMR_EMAX_EV` | `1.0` | Maximum neutron energy processed by THERMR. |
| `THERMR_TOLERANCE` | `0.01` | Numerical tolerance for THERMR. |
| `THERMR_ANGULAR_BINS` | `16` | Number of THERMR angular bins. |

**NCrystal**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ENERGY_MIN_EV` | `1e-5` | First incident neutron energy in eV. |
| `ENERGY_MAX_EV` | `1.0` | Last incident neutron energy in eV. |

In [ ]:
if ENGINE == "njoy":
    THERMR_EMAX_EV = 1.0
    THERMR_TOLERANCE = 0.01
    THERMR_ANGULAR_BINS = 16
elif ENGINE == "ncrystal":
    ENERGY_MIN_EV = 1e-5
    ENERGY_MAX_EV = 1.0

In [ ]:
import numpy as np

if ENGINE == "njoy":
    from endf_parserpy import EndfParserPy
    from input_output.leapr.workflows import serialize_leapr_material
    from processing.backends.njoy_processor import NjoyProcessor
    from processing.deck_builders.leapr_builder import build_analysis_deck
elif ENGINE == "ncrystal":
    from dataclasses import replace
    import re
    from tempfile import TemporaryDirectory

    from NCrystal.plugins import browsePlugins
    from input_output.ncrystal.liquids import render_custom_liquids
    from processing.backends.ncrystal_processor import NcrystalProcessor

In [ ]:
cross_sections = {}

if ENGINE == "njoy":
    if NJOY_EXECUTABLE is None:
        raise FileNotFoundError("Set NJOY_EXEC to the executable path or add njoy to PATH.")
    parser = EndfParserPy()
    base_file = str(BASE_ENDF_FILE)
    base_header = parser.parsefile(base_file, include=[(1, 451)])[1][451]
    if int(base_header["ZA"]) != 1001:
        raise ValueError("The CAB hydrogen example requires an H-1 neutron evaluation.")
    if int(template.principal_scatterer["iel"]) != 0:
        raise ValueError("This comparison expects the CAB template without a separate elastic reaction.")
    processor = NjoyProcessor(njoy_exec=NJOY_EXECUTABLE)

    for sample_id, realization in realizations.items():
        module_text = serialize_leapr_material(template, realization)
        deck = build_analysis_deck(
            base_mat=int(base_header["MAT"]),
            leapr_mat=int(template.material_info["mat"]),
            temperature=TEMPERATURE_K,
            leapr_module_text=module_text,
            thermr_emax=THERMR_EMAX_EV,
            thermr_tol=THERMR_TOLERANCE,
            thermr_nbin=THERMR_ANGULAR_BINS,
            thermr_natom=int(template.principal_scatterer["npr"]),
        )
        try:
            with processor.execute(deck, input_tapes={"tape20": base_file}) as execution:
                if not execution.has_tape("tape26"):
                    raise RuntimeError(f"Missing THERMR tape26.\n{execution.stdout_text}")
                table = parser.parsefile(str(execution.tape_path("tape26")), include=[3])
                xs_table = table[3][221]["xstable"]
                energy = np.asarray(xs_table["E"], dtype=float)
                xs = np.asarray(xs_table["xs"], dtype=float)
                if (
                    energy.ndim != 1 or energy.size < 2 or xs.shape != energy.shape
                    or not np.isfinite(energy).all() or not np.isfinite(xs).all()
                    or np.any(energy <= 0) or np.any(np.diff(energy) < 0)
                    or np.any(xs < 0)
                ):
                    raise ValueError("Invalid THERMR MF3/MT221 cross-section table.")
                cross_sections[sample_id] = (energy, xs)
        except (RuntimeError, OSError, KeyError, ValueError) as error:
            raise RuntimeError(f"Cross-section calculation failed for {sample_id}.") from error
        print(f"{sample_id}: calculated thermal scattering cross section")
elif ENGINE == "ncrystal":
    if not any(name == "Liquids" for name, _, _ in browsePlugins()):
        raise RuntimeError("Load the NCrystal Liquids plugin before launching Jupyter.")
    energy_grid = np.geomspace(ENERGY_MIN_EV, ENERGY_MAX_EV, 500)
    liquid_block = re.compile(r"^@CUSTOM_LIQUIDS[^\n]*\n.*?(?=^@|\Z)", re.MULTILINE | re.DOTALL)
    processor = NcrystalProcessor()

    with TemporaryDirectory(prefix="tara_ncrystal_translation_") as directory:
        for sample_id, realization in realizations.items():
            text = source_text
            if sample_id != "sample_000":
                dynamics = realization.dynamics
                sampled_hydrogen = replace(
                    hydrogen,
                    wt=dynamics.translation_weight,
                    ws=dynamics.continuous_weight,
                    c=dynamics.translation_parameter,
                )
                entries = [
                    sampled_hydrogen if entry.atom == "H" else entry for entry in liquid_entries
                ]
                rendered = render_custom_liquids(entries)
                text, replacements = liquid_block.subn(lambda _: rendered, source_text)
                if replacements != 1:
                    raise ValueError("Expected one @CUSTOM_LIQUIDS block to replace.")
            sample_path = Path(directory) / f"{sample_id}.ncmat"
            sample_path.write_text(text, encoding="utf-8")

            state = processor.create_native_scattering_state(
                str(sample_path), TEMPERATURE_K, verify_plugin=False,
            )
            cross_sections[sample_id] = (energy_grid, processor.native_cross_sections(state, energy_grid))
            print(f"{sample_id}: calculated native material scattering cross section")

## 7. Compare the calculated cross sections

The top panel shows the curves over their common energy range. The bottom
panel zooms into the shaded region and shows
`100 × (sample − reference) / reference`. All curves use the same energy grid
in each panel.

**Optional settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `LOW_ENERGY_MAX_EV` | `1e-2` | Upper edge of the low-energy comparison in eV. |

In [ ]:
LOW_ENERGY_MAX_EV = 1e-2

In [ ]:
import matplotlib.pyplot as plt

energy_min = max(energy[0] for energy, _ in cross_sections.values())
energy_max = min(energy[-1] for energy, _ in cross_sections.values())

if ENGINE == "njoy":
    energy_max = min(THERMR_EMAX_EV, energy_max)
    cross_section_label = "Thermal scattering cross section [barn]"
elif ENGINE == "ncrystal":
    cross_section_label = "Native scattering cross section [barn/atom]"

zoom_max = min(LOW_ENERGY_MAX_EV, energy_max)
if zoom_max <= energy_min:
    raise ValueError("Choose a zoom limit above the first common evaluation energy.")

energy_grid = np.geomspace(energy_min, energy_max, 500)
zoom_grid = np.geomspace(energy_min, zoom_max, 250)
reference_energy, reference_xs = cross_sections["sample_000"]
reference_zoom = np.interp(zoom_grid, reference_energy, reference_xs)
valid_reference = reference_zoom > 0
if not valid_reference.any():
    raise ValueError("The reference cross section is zero throughout the selected zoom.")
fig, (full_ax, zoom_ax) = plt.subplots(2, 1, figsize=(10, 7), layout="constrained")
for index, (sample_id, (energy, xs)) in enumerate(cross_sections.items()):
    is_reference = sample_id == "sample_000"
    color = "black" if is_reference else f"C{index - 1}"
    label = "sample_000 (reference)" if is_reference else sample_id
    full_ax.loglog(
        energy_grid, np.interp(energy_grid, energy, xs),
        color=color, linewidth=2 if is_reference else 1.4, label=label,
    )
    if not is_reference:
        sample_zoom = np.interp(zoom_grid, energy, xs)
        relative = np.full_like(reference_zoom, np.nan)
        np.divide(
            100 * (sample_zoom - reference_zoom), reference_zoom,
            out=relative, where=valid_reference,
        )
        zoom_ax.semilogx(zoom_grid, relative, color=color, label=sample_id)
full_ax.axvspan(energy_min, zoom_max, color="0.9", alpha=0.45, zorder=0)

full_ax.set(xlabel="Incident neutron energy [eV]", ylabel=cross_section_label)

full_ax.legend()
zoom_ax.axhline(0, color="black", linestyle="--", linewidth=1, label="Reference")
zoom_ax.set(
    xlabel="Incident neutron energy [eV]", ylabel="Relative difference [%]",
    title="Low incident energy",
)
for axis in (full_ax, zoom_ax):
    axis.grid(True, alpha=0.25)
plt.show()

Compare the weight factors in Step 5 with the low-energy differences. The diffusivity factor is 1 for every sample.

See also [Diffusivity at fixed translation weight](diffusivity_effect.ipynb) and [Joint translation sampling](joint_translation_effect.ipynb).

[Cross-section analysis](../02_analysis/cross_sections/liquid.ipynb) · [Liquid ACE generation](../03_ace_generation/liquid.ipynb)